In [1]:
"""
MAP - Charting Student Math Misunderstandings - Inference Notebook v4 (Hybrid)
Phase 1: beam search 取 top-K 候選
Phase 2: 過濾到訓練集 unique labels,再用 log-likelihood 重排
Phase 3: 取 top-3,fallback 補齊
"""
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from tqdm import tqdm

# ==== 路徑 ====
BASE_MODEL_PATH = "/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1"
ADAPTER_PATH = "/kaggle/input/datasets/alextsai2004/gemma-math-misunderstanding-lora/best_gemma_lora_model"
TEST_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/test.csv"
TRAIN_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/train.csv"
SAMPLE_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/sample_submission.csv"
OUTPUT_CSV = "/kaggle/working/submission.csv"

# ==== 超參 ====
BEAM_BATCH_SIZE = 16     # 同時 beam search 多少筆 test
NUM_BEAMS = 10            # beam search 寬度
NUM_RETURN = 10           # 每筆要拿多少候選
MAX_NEW_TOKENS = 32


# ==== Sample submission 當骨架 ====
print("=" * 60)
sample = pd.read_csv(SAMPLE_CSV)
print(f"Sample shape: {sample.shape}")
ROW_ID_COL = sample.columns[0]
PRED_COL = sample.columns[1]


# ==== Prompt ====
def build_user_prompt(question, correct_answer, student_explanation):
    return (
        "You are a math misconception classifier.\n"
        "Given the question, the correct answer, and the student's explanation, "
        "predict the final label in the format `Category:Misconception`.\n"
        "If the category is not a misconception type, use `NA` for the misconception part.\n\n"
        f"Question: {question}\n"
        f"Correct answer: {correct_answer}\n"
        f"Student explanation: {student_explanation}\n\n"
        "Return only the label."
    )


def build_prompt(row):
    user = build_user_prompt(
        row["QuestionText"], row["MC_Answer"], row["StudentExplanation"]
    )
    return f"<start_of_turn>user\n{user}<end_of_turn>\n<start_of_turn>model\n"


# ==== Model ====
print("Loading base model in bf16 ...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
device = next(model.parameters()).device
pad_id = tokenizer.pad_token_id

end_of_turn_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")
stop_ids = [tokenizer.eos_token_id]
if end_of_turn_id and end_of_turn_id != tokenizer.unk_token_id:
    stop_ids.append(end_of_turn_id)


# ==== 訓練集 labels (用於 filter 和 fallback) ====
train_df = pd.read_csv(TRAIN_CSV)
train_df["target"] = (
    train_df["Category"].astype(str) + ":" +
    train_df["Misconception"].fillna("NA").astype(str)
)
unique_labels = sorted(train_df["target"].unique().tolist())
unique_labels_set = set(unique_labels)
fallback_labels = train_df["target"].value_counts().head(3).index.tolist()
print(f"# unique labels: {len(unique_labels)}")
print(f"Fallback labels: {fallback_labels}")


# ==== Test ====
test_df = pd.read_csv(TEST_CSV)
for col in ["QuestionText", "MC_Answer", "StudentExplanation"]:
    test_df[col] = test_df[col].fillna("")
print(f"Test size: {len(test_df)}")
assert len(test_df) == len(sample)


# ==== Helper ====
def clean_label(text):
    """從 beam 輸出抽出乾淨的 label 字串"""
    if not text:
        return ""
    label = text.splitlines()[0].strip()
    if " " in label:
        label = label.split(" ")[0]
    return label


# ==== Phase 1: 批次 beam search 拿候選 ====
@torch.no_grad()
def beam_generate_batch(prompts):
    """回傳每個 prompt 的 NUM_RETURN 個候選 label (filter + dedup 後)"""
    enc = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=1024,
    ).to(device)
    outputs = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        num_return_sequences=NUM_RETURN,
        do_sample=False,
        early_stopping=True,
        eos_token_id=stop_ids,
        pad_token_id=tokenizer.pad_token_id,
    )
    prompt_len = enc["input_ids"].shape[1]
    outputs = outputs.view(len(prompts), NUM_RETURN, -1)

    results = []
    for i in range(len(prompts)):
        valid = []
        seen = set()
        for k in range(NUM_RETURN):
            gen = outputs[i, k, prompt_len:]
            text = tokenizer.decode(gen, skip_special_tokens=True).strip()
            label = clean_label(text)
            # 只保留訓練集出現過的 label,且去重
            if label in unique_labels_set and label not in seen:
                valid.append(label)
                seen.add(label)
        results.append(valid)
    return results


# ==== Phase 2: log-likelihood 重排候選 ====
@torch.no_grad()
def score_candidates(prompt_text, candidate_labels):
    """對一個 prompt 的候選 labels 算 log-likelihood,回傳分數 list"""
    if not candidate_labels:
        return []

    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=True)
    candidate_tokens = [
        tokenizer.encode(c + "<end_of_turn>", add_special_tokens=False)
        for c in candidate_labels
    ]

    B = len(candidate_tokens)
    sequences = [prompt_ids + tok for tok in candidate_tokens]
    max_len = max(len(s) for s in sequences)

    input_ids = torch.full((B, max_len), pad_id, dtype=torch.long)
    attention_mask = torch.zeros((B, max_len), dtype=torch.long)
    label_starts = []
    for j, seq in enumerate(sequences):
        pad = max_len - len(seq)
        input_ids[j, pad:] = torch.tensor(seq, dtype=torch.long)
        attention_mask[j, pad:] = 1
        label_starts.append(max_len - len(candidate_tokens[j]))

    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits

    scores = []
    for j, tok in enumerate(candidate_tokens):
        L = len(tok)
        ls = label_starts[j]
        slice_logits = logits[j, ls - 1:ls - 1 + L, :].float()
        log_probs = torch.log_softmax(slice_logits, dim=-1)
        target = torch.tensor(tok, device=device)
        tok_lp = log_probs.gather(1, target.unsqueeze(1)).squeeze(1)
        scores.append(tok_lp.mean().item())  # length-normalized
    return scores


# ==== Phase 1 執行: 收集所有候選 ====
print("\n[Phase 1] Beam search ...")
all_candidates = {}  # row_id -> list of valid candidate labels
for start in tqdm(range(0, len(test_df), BEAM_BATCH_SIZE), desc="Beam"):
    batch = test_df.iloc[start:start + BEAM_BATCH_SIZE]
    prompts = [build_prompt(r) for _, r in batch.iterrows()]
    batch_results = beam_generate_batch(prompts)
    for (_, row), cands in zip(batch.iterrows(), batch_results):
        all_candidates[row["row_id"]] = cands


# ==== Diagnostic: 看 beam 召回率 ====
n_zero = sum(1 for v in all_candidates.values() if len(v) == 0)
n_lt3 = sum(1 for v in all_candidates.values() if len(v) < 3)
avg_cands = np.mean([len(v) for v in all_candidates.values()])
print(f"\nBeam diagnostics:")
print(f"  Avg valid candidates per row: {avg_cands:.2f}")
print(f"  Rows with 0 valid candidates: {n_zero} ({n_zero/len(test_df)*100:.1f}%)")
print(f"  Rows with <3 valid candidates: {n_lt3} ({n_lt3/len(test_df)*100:.1f}%)")


# ==== Phase 2 執行: log-likelihood 重排 ====
print("\n[Phase 2] Log-likelihood re-rank ...")
pred_dict = {}
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Rerank"):
    rid = row["row_id"]
    candidates = all_candidates[rid]

    if len(candidates) == 0:
        top3 = list(fallback_labels[:3])
    else:
        prompt = build_prompt(row)
        scores = score_candidates(prompt, candidates)
        ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
        top3 = [c for c, _ in ranked]
        # 補滿 3 個
        for fb in fallback_labels:
            if len(top3) >= 3:
                break
            if fb not in top3:
                top3.append(fb)
        while len(top3) < 3:
            top3.append(top3[0])

    pred_dict[rid] = " ".join(top3[:3])


# ==== 用 sample 骨架建 submission ====
submission = sample.copy()
submission[PRED_COL] = submission[ROW_ID_COL].map(pred_dict)

print("\n" + "=" * 60)
print("Validation:")
print(f"Shape: {submission.shape}")
print(f"Any NaN: {submission.isna().any().any()}")
print(f"Head:\n{submission.head()}")

assert submission.shape == sample.shape
assert not submission.isna().any().any()
assert (submission[PRED_COL] != "").all()
assert (submission[PRED_COL].str.split().str.len() == 3).all()

submission.to_csv(OUTPUT_CSV, index=False)
print(f"\n[OK] Saved {OUTPUT_CSV}")

Sample shape: (3, 2)
Loading base model in bf16 ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


# unique labels: 65
Fallback labels: ['True_Correct:NA', 'False_Neither:NA', 'True_Neither:NA']
Test size: 3

[Phase 1] Beam search ...


Beam: 100%|██████████| 1/1 [00:05<00:00,  5.17s/it]



Beam diagnostics:
  Avg valid candidates per row: 5.33
  Rows with 0 valid candidates: 0 (0.0%)
  Rows with <3 valid candidates: 1 (33.3%)

[Phase 2] Log-likelihood re-rank ...


Rerank: 100%|██████████| 3/3 [00:02<00:00,  1.35it/s]


Validation:
Shape: (3, 2)
Any NaN: False
Head:
   row_id                             Category:Misconception
0   36696  True_Neither:NA True_Correct:NA True_Misconcep...
1   36697  False_Neither:NA False_Misconception:WNB False...
2   36698   True_Neither:NA True_Correct:NA False_Neither:NA

[OK] Saved /kaggle/working/submission.csv
